# Navigating the Grey: every demo, step by step

**PyCon Davao 2026 · DurianPy** · Alpha Romer Coma
*Building resilient Python scrapers that survive at scale*

Parts 1 to 3 are the talk's live demos, in the order the slides show them. Part 4 is a bonus the talk doesn't run on stage. Run **Part 0** first, then go top to bottom, or jump to any part.

| Part | What you'll see | Requests to real sites | Time |
|---|---|---|---|
| 0 · Setup | installs, the talk's code, the recorded pages, a browser | none | ~2 min |
| 1 · Where the data lives | robots.txt, then three doors: HTML, a browser, the page's own API | 5 robots.txt files, 1 Robinsons page, 1 Megaworld page (and three times in a browser, like any visitor), 3 API calls | ~3 min |
| 2 · Fast | 53 malls: a for loop, asyncio, threads, processes, both, with a scoreboard | **none**: a replay of 53 recorded pages, on this machine | ~2 min |
| 3 · Polite | a limit (on a test server), then your IP: SM direct and through proxies | 1 to SM; with your own proxies, +1 per proxy and about 11 more | ~2 min |
| 4 · PengePassportPH (bonus) | the scraping inside the package, by hand: token, an empty 200, dates, hours. Then the package | **none** by default (a stand-in on this machine); about 15 to passport.gov.ph if you tick LIVE | ~2 min |

**Cells marked 🌐 send real requests to someone else's server.** Each is counted above and spaced out. Run each one once, and don't put them in a loop.

**Runtime:** the default CPU runtime is enough. Colab's free runtime has 2 cores, like the machine behind the slide numbers, so your times should land close to the talk's.

**Presenting from it?** Run Part 0 before you go on, then **Edit → Clear all outputs** so every cell runs fresh. The servers Part 0 starts keep running.

## Part 0 · Setup
Run all four cells. They install the libraries, unpack the talk's scripts and the 53 Robinsons mall pages recorded on 14 Oct 2026 (so Part 2 never touches Robinsons), install a browser for door 2, and start two small servers on this machine.

In [ ]:
%pip install -q requests beautifulsoup4 aiohttp protego pandas playwright
%pip install -q "penge-passport-ph @ git+https://github.com/alpharomercoma/penge-passport-ph.git@2d99430ac66aad5d1d5355a4dd87571e86fdce9c#subdirectory=packages/python"

In [ ]:
#@title 0.2 · Unpack the talk's code and the recorded pages
import base64, io, lzma, os, sys, tarfile, json
from pathlib import Path

BLOB = """
...
ANDLocQUFzswAwAAAAAEWVo=
"""
ROOT = (Path("/content") if Path("/content").is_dir() else Path.cwd()) / "navigating-the-grey"
ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:xz") as tar:
    try:
        tar.extractall(ROOT, filter="data")
    except TypeError:                      # Python without extraction filters
        tar.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

pages = list((ROOT / "data/recordings/robinsons").glob("*.html"))
size = sum(p.stat().st_size for p in pages) / 1e6
print(f"Unpacked to {ROOT}")
print(f"  {len(pages)} Robinsons mall pages ({size:.1f} MB), recorded 14 Oct 2026 with how long each took")
print(f"  code: {', '.join(sorted(str(p.relative_to(ROOT)) for p in ROOT.glob('*/*.py') if p.name != '__init__.py'))}")

In [ ]:
#@title 0.3 · Install Chromium for door 2 (on Colab, about a minute)
import subprocess
if "google.colab" in sys.modules:
    done = subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"],
                          capture_output=True, text=True)
    print("Chromium installed." if done.returncode == 0 else
          "The Chromium install failed; door 2's browser cell will say so too:\n" + done.stderr[-1500:])
else:
    print("Not on Colab. If Chromium is missing, run: python -m playwright install chromium")

In [ ]:
#@title 0.4 · Start the two local servers
import os, sys, time, json, warnings
from pathlib import Path
ROOT = next((p for p in (Path("/content/navigating-the-grey"), Path.cwd() / "navigating-the-grey", Path.cwd())
             if (p / "common" / "malls.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run Part 0 (Setup) first.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"   # for !penge
warnings.filterwarnings("ignore", message=r".*fork\(\) may lead to deadlocks")       # see Part 2.5
from notebook_helpers import (HEADERS, ensure_replay, ensure_server, record, score, scoreboard,
                              show_results, timeline)

print("replay     http://127.0.0.1:8800  the 53 recorded Robinsons pages:", "up" if ensure_replay() else "DIDN'T START")
print("stand-in   http://127.0.0.1:8802  a stand-in for passport.gov.ph:  ",
      "up" if ensure_server("passport_standin.py", 8802) else "DIDN'T START")
print("Ready:", ROOT)

## Part 1 · Where does the data live?
*Slides: Where Does the Data Live? → Door 1 (the page, then the code) → Door 2 (the page, then Playwright) → Door 3*

Same kind of data, three doors: the stores are in the HTML (**door 1**), the page is empty until JavaScript runs (**door 2**), or the JSON the page itself fetches (**door 3**). And before any door: read robots.txt.

In [ ]:
#@title Part 1 needs Part 0: check
import os, sys, time, json, warnings
from pathlib import Path
ROOT = next((p for p in (Path("/content/navigating-the-grey"), Path.cwd() / "navigating-the-grey", Path.cwd())
             if (p / "common" / "malls.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run Part 0 (Setup) first.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"   # for !penge
warnings.filterwarnings("ignore", message=r".*fork\(\) may lead to deadlocks")       # see Part 2.5
from notebook_helpers import (HEADERS, ensure_replay, ensure_server, record, score, scoreboard,
                              show_results, timeline)
print("Ready:", ROOT)

### 1.1 · Read robots.txt before the page 🌐 5 requests
`protego` reads robots.txt the way search engines do, wildcards included. One robots.txt per site. Under the robots.txt standard ([RFC 9309](https://www.rfc-editor.org/rfc/rfc9309)), a site that answers 4xx for it has set no rules.

In [ ]:
import requests
from urllib.parse import urlsplit
from protego import Protego

PAGES = [
    ("Robinsons, a mall page",         "https://robinsonsmalls.com/mall-info/robinsons-cybergate-davao"),
    ("Megaworld, a mall page",         "https://www.megaworld-lifestylemalls.com/malls/uptown-mall"),
    ("Megaworld's API (Contentstack)", "https://cdn.contentstack.io/v3/content_types/tenant/entries/"),
    ("SM, the mall list",              "https://www.smsupermalls.com/list-of-malls?region=all&keyword=&page=0"),
    ("SM, a store directory",          "https://www.smsupermalls.com/mall-directory/sm-city-davao/information/shopping/"),
    ("Ayala, Abreeza's directory",     "https://www.ayalamalls.com/explore/mall-directory/ayala-abreeza"),
]

robots = {}
for name, url in PAGES:
    host = urlsplit(url).netloc
    if host not in robots:                                   # one robots.txt per site
        try:
            res = requests.get(f"https://{host}/robots.txt", headers=HEADERS, timeout=20)
            robots[host] = (res.status_code, Protego.parse(res.text) if res.ok else None)
        except requests.RequestException as e:
            robots[host] = (type(e).__name__, None)
    status, rules = robots[host]
    if rules:
        verdict = "✓ allowed" if rules.can_fetch(url, "*") else "✗ disallowed"
    elif isinstance(status, int) and 400 <= status < 500 and status != 429:
        verdict = f"✓ no rules (HTTP {status})"
    else:
        verdict = f"? unreadable ({status})"
    print(f"{verdict:<22} {name}")

Robinsons and Megaworld allow bots, which is why the demos use them. SM keeps bots off its store directory pages (`*/information/shopping/*`), and Ayala off its whole site.

Megaworld's API host sets no rules, so the letter allows it. Whether a site's own API is fair game is the grey part: that judgment is yours.

### 1.2 · Door 1: the stores are in the HTML 🌐 1 request
*Slides: Door 1 · Finding the Selector → Door 1 · Requests + BeautifulSoup*

Right-click a store on [the live page](https://robinsonsmalls.com/mall-info/robinsons-cybergate-davao) and choose Inspect: every store is an `li` whose class includes `store-name`. `requests` downloads the page; BeautifulSoup picks elements with CSS selectors. Every store is an `li` with class `store-name`: the name, a bar, the phone. (Look at Eye Level's phone: real pages are messy, and the table keeps what the page says.)

In [ ]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://robinsonsmalls.com/mall-info/"
mall = "robinsons-cybergate-davao"

t = time.perf_counter()
try:
    res = requests.get(URL + mall, headers=HEADERS, timeout=30)
    res.raise_for_status()
    html, how = res.text, f"in {time.perf_counter() - t:.1f} s"
except requests.RequestException as e:   # offline, or refused: use the copy recorded on 14 Oct
    html = Path(f"data/recordings/robinsons/{mall}.html").read_text(encoding="utf-8")
    how = f"(the 14 Oct recording, because the live request failed: {type(e).__name__})"

soup = BeautifulSoup(html, "html.parser")
rows = []
for li in soup.select("li.store-name"):
    name, _, phone = li.text.partition("|")
    rows.append({"name": name.strip(),
                 "phone": phone.strip()})

print(f"What comes back: {len(html.encode()) / 1000:.0f} KB of HTML {how}")
print(f"What we get: {len(rows)} stores")
pd.DataFrame(rows).head()

### 1.3 · Door 2: empty until JavaScript runs 🌐 1 page, then the same page in a browser three times (about 140 requests each, like any visitor)
*Slides: Door 2 · Uptown Mall in a Browser → Door 2 · Playwright → Door 2 · Headful or Headless?*

In [a browser](https://www.megaworld-lifestylemalls.com/malls/uptown-mall) the stores are there; Inspect shows each one as `a.store-item` with its name in `.store-item-name`. View Source shows what `requests` gets. First, that. Then a real browser (Playwright) runs the page's JavaScript. Then the same page headless and headful, side by side.

In [ ]:
import time, requests

MEGA = "https://www.megaworld-lifestylemalls.com/malls/uptown-mall"

t = time.perf_counter()
try:
    shell = requests.get(MEGA, headers=HEADERS, timeout=30).text
    print(f"What requests sees: {len(shell.encode()) / 1000:.1f} KB, "
          f"{shell.count('store-item-name')} stores, in {time.perf_counter() - t:.1f} s")
    if '<div id="root"></div>' in shell.replace("\n", ""):
        print('  <div id="root"></div>      <- the stores go here, after JavaScript runs')
except requests.RequestException as e:
    print(f"Couldn't reach Megaworld ({type(e).__name__}). On 15 Oct it sent a 3 KB page with 0 stores.")

In [ ]:
from playwright.async_api import async_playwright

# only needed behind a corporate or cloud proxy; Colab has none
proxy = {"proxy": {"server": os.environ["HTTPS_PROXY"]}} if os.environ.get("HTTPS_PROXY") else {}

seen, names, t = [], [], time.perf_counter()
try:
    async with async_playwright() as p:
        browser = await p.chromium.launch(**proxy)       # headless
        page = await browser.new_page()
        page.on("request", lambda r: seen.append(r))
        await page.goto(MEGA)                            # Uptown Mall
        await page.wait_for_selector(".store-item")
        names = await page.locator(".store-item-name").all_inner_texts()   # after JS
        await browser.close()
    print(f"What Playwright sees: {len(names)} stores in {time.perf_counter() - t:.1f} s "
          f"(browser start included), after the page made {len(seen)} requests")
except Exception as e:
    print(f"The browser run failed ({type(e).__name__}: {str(e).splitlines()[0][:120]}).")
    print("On 15 Oct the same run found 10 stores in 11 s, after the page made 139 requests.")
names[:5]

In [ ]:
#@title Headful or headless? The same page twice more: the time, the memory, what the page sees
import shutil, subprocess, psutil

# headful needs a screen. Colab has none, so give it a virtual one: Xvfb, installed with Chromium in 0.3
if not os.environ.get("DISPLAY") and shutil.which("Xvfb"):
    subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1280x800x24"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.environ["DISPLAY"] = ":99"
    time.sleep(1)

def browser_mb():
    """Memory of the browser's own processes, not this notebook's."""
    total = 0
    for proc in psutil.Process().children(recursive=True):
        try:
            if "chrom" in proc.name() or "headless" in proc.name():
                total += proc.memory_full_info().pss
        except psutil.Error:
            pass
    return total / 1e6

async def door2(**how):
    t = time.perf_counter()
    async with async_playwright() as p:
        browser = await p.chromium.launch(**how, **proxy)
        page = await browser.new_page()
        await page.goto(MEGA)
        await page.wait_for_selector(".store-item")
        stores = len(await page.locator(".store-item-name").all_inner_texts())
        seconds, mb = time.perf_counter() - t, browser_mb()
        agent, webdriver = await page.evaluate(
            "() => [navigator.userAgent.match(/(Headless)?Chrome/)[0], navigator.webdriver]")
        await browser.close()
    return f"{stores} stores in {seconds:4.1f} s, {mb:3.0f} MB. The page sees {agent}, webdriver={webdriver}"

for label, how in [("headless  launch()", {}), ("headful   launch(headless=False)", {"headless": False})]:
    try:
        print(f"{label:<34} {await door2(**how)}")
    except Exception as e:
        print(f"{label:<34} failed ({type(e).__name__}: {str(e).splitlines()[0][:100]})")
    time.sleep(2)
print("\nOn the slide (16 Oct, median of 3): headless 10.1 s, 370 MB; headful 6.5 s, 520 MB.")

A notebook already runs an event loop, so these cells use Playwright's async API with a bare `await`. In a script you'd use the sync API, as on the slide. To watch the window yourself, run `python -m doors.door2_megaworld --headful` from the repo on a computer with a screen.

### 1.4 · Door 3: the page's own API 🌐 3 requests, 1 s apart
*Slide: Door 3 · The Page's Own API*

Megaworld's page asks a content service (Contentstack) for its stores. Ask it yourself. The keys are the public, read-only ones the site sends to every visitor's browser.

In [ ]:
import time, requests

API = "https://cdn.contentstack.io/v3/content_types/tenant/entries/"
KEYS = {"api_key": "blt827157d7af7bc6d4", "access_token": "cs12c8f62754c81457af4cc5fc"}
q = {"environment": "prod-environment", "limit": 100,
     "query": '{"reference": {"$in_query": {"name": "Uptown Mall"}}}'}

def ask(params):
    res = requests.get(API, params=params, headers=HEADERS | KEYS, timeout=30)
    res.raise_for_status()
    return res.json()["entries"]

try:
    t = time.perf_counter()
    rows = ask(q)               # 100 stores
    print(f"What comes back: JSON in {time.perf_counter() - t:.1f} s, {len(rows)} stores in one request")
    print(" ", json.dumps({k: rows[0][k] for k in ("name", "category", "shop_type", "floor")}, ensure_ascii=False))

    # the whole mall: keep asking until a page comes back short
    everything, skip = list(rows), len(rows)
    while len(rows) == 100:
        time.sleep(1)           # politely
        rows = ask(q | {"skip": skip})
        everything += rows
        skip += len(rows)
    print(f"\nAll of Uptown Mall: {len(everything)} stores in {-(-len(everything) // 100)} requests")
except (requests.RequestException, KeyError, ValueError) as e:
    print(f"The API didn't answer as expected ({type(e).__name__}). "
          "On 15 Oct: 100 stores in one request, all 248 in 3.")

Same mall, three ways: door 1 parses HTML, door 2 needs a whole browser and about 140 requests to show the first 10 stores, door 3 gets 100 stores per request as clean JSON, all 248 in three. Look for door 3 first.

## Part 2 · Fast: all 53 malls, six ways
*Slides: Where the Time Goes → Sync vs Async → Threads vs Processes → All 53 Malls, Six Ways*

Every request in this part goes to `replay/server.py`, a small server on this machine. It answers with the 53 Robinsons pages recorded on 14 Oct 2026, each after the delay the real site took then. **Robinsons sees zero requests from this part.** (The delays were recorded from a cloud server, not from Davao.)

Each run lands on a scoreboard next to the slide's number. The slide's numbers are medians of 3 runs; here it's one run each, so expect a little wobble.

In [ ]:
#@title Part 2 needs Part 0: check (and start the replay)
import os, sys, time, json, warnings
from pathlib import Path
ROOT = next((p for p in (Path("/content/navigating-the-grey"), Path.cwd() / "navigating-the-grey", Path.cwd())
             if (p / "common" / "malls.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run Part 0 (Setup) first.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"   # for !penge
warnings.filterwarnings("ignore", message=r".*fork\(\) may lead to deadlocks")       # see Part 2.5
from notebook_helpers import (HEADERS, ensure_replay, ensure_server, record, score, scoreboard,
                              show_results, timeline)

print("The replay is answering at http://127.0.0.1:8800/mall-info/<slug>" if ensure_replay()
      else "The replay server didn't start. Run Part 0 again, then this cell.")
print("Ready:", ROOT)

### 2.1 · All 53 malls, one at a time
*Slide: Where the Time Goes.* About half a minute: talk over it.

The plain for loop. It also times the two halves of every mall: waiting for the page, and parsing it.

In [ ]:
import statistics
import requests
from doors.door1_robinsons import parse  # door 1's loop, as a function
from common.malls import SLUGS, URL      # the 53 malls; URL is the replay on this machine

waits, parses = [], []

def scrape_mall(slug):
    t0 = time.perf_counter()
    res = requests.get(URL + slug)       # waiting: the CPU sits idle
    t1 = time.perf_counter()
    rows = parse(res.text)               # parsing: the CPU works
    waits.append(t1 - t0)
    parses.append(time.perf_counter() - t1)
    return rows

started = time.perf_counter()
pages = [scrape_mall(m) for m in SLUGS]
score("Sync, a for loop", started, pages)

print(f"\nOne page, the median: {statistics.median(waits) * 1000:.0f} ms waiting, "
      f"{statistics.median(parses) * 1000:.0f} ms parsing")
print(f"All 53: {sum(waits):.1f} s waiting, {sum(parses):.1f} s parsing. "
      f"{sum(waits) / (sum(waits) + sum(parses)):.0%} of it is waiting.")

Waiting is **I/O-bound**: the CPU sits idle. Parsing is **CPU-bound**. Two different fixes: overlap the waiting, or use more cores.

### 2.2 · Sync vs async, three malls
*Slide: Sync vs Async, Side by Side.* Three malls near Davao. Sync asks for one, waits, parses, then asks for the next.

In [ ]:
import requests
from doors.door1_robinsons import parse
from common.malls import URL

THREE = ["robinsons-cybergate-davao", "robinsons-place-tagum", "robinsons-place-gensan"]
log = []

def scrape_mall(slug):
    sent = time.perf_counter()
    res = requests.get(URL + slug)
    got = time.perf_counter()
    rows = parse(res.text)
    log.append((slug, sent, got, time.perf_counter()))
    return rows

start = time.perf_counter()
pages = [scrape_mall(m) for m in THREE]
print(f"Sync: 3 malls in {time.perf_counter() - start:.2f} s\n")
timeline(log, start)

Async: the same steps with `async def` and `await`. At every `await` the one thread stops waiting and starts the next mall, so the waits overlap. Parsing still happens one page at a time, so a page that arrives while another is being parsed waits its turn.

In [ ]:
import asyncio
import aiohttp

log = []

async def scrape_mall(s, slug):
    sent = time.perf_counter()
    res = await s.get(URL + slug)
    html = await res.text()
    got = time.perf_counter()
    rows = parse(html)
    log.append((slug, sent, got, time.perf_counter()))
    return rows

start = time.perf_counter()
async with aiohttp.ClientSession() as s:
    jobs = [scrape_mall(s, m) for m in THREE]
    pages = await asyncio.gather(*jobs)
print(f"Async: 3 malls in {time.perf_counter() - start:.2f} s\n")
timeline(log, start)

### 2.3 · All 53 with asyncio
The same async function on all 53 malls, with a semaphore so no more than 32 are in flight at once: we don't flood the server.

In [ ]:
import asyncio, aiohttp
from doors.door1_robinsons import parse
from common.malls import SLUGS, URL

at_once = asyncio.Semaphore(32)          # 32 in flight, at most

async def scrape_mall(s, slug):
    async with at_once:
        res = await s.get(URL + slug)
        return parse(await res.text())

started = time.perf_counter()
async with aiohttp.ClientSession() as s:
    jobs = [scrape_mall(s, m) for m in SLUGS]
    pages = await asyncio.gather(*jobs)
score("asyncio, 32 at once", started, pages)

### 2.4 · All 53 with 32 threads
*Slide: Threads vs Processes.* No `async`: keep the plain sync function and run it in a pool of 32 threads. The waits overlap, but the GIL lets one thread run Python at a time, so the parsing still takes turns on one core.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from common.malls import SLUGS
from scale.step1_sync import scrape_mall   # the sync one, unchanged

started = time.perf_counter()
with ThreadPoolExecutor(32) as pool:
    pages = list(pool.map(scrape_mall, SLUGS))
score("32 threads", started, pages)

### 2.5 · All 53 with 32 processes
One word changes. Each worker is a whole Python, so the parsing runs on every core. Each worker imports `scrape_mall` from its file, which is why it lives in `scale/step1_sync.py` and not in a cell. In a script, the pool goes under `if __name__ == "__main__":`.

In [ ]:
import gc, multiprocessing
from concurrent.futures import ProcessPoolExecutor
from common.malls import SLUGS
from scale.step1_sync import scrape_mall   # the sync one, unchanged

gc.freeze()                              # a notebook-only tweak: see below
started = time.perf_counter()
with ProcessPoolExecutor(32) as pool:
    pages = list(pool.map(scrape_mall, SLUGS))
score("32 processes", started, pages)
print(f"{os.cpu_count()} cores, workers started by {multiprocessing.get_start_method()}")

`gc.freeze()` is for notebooks. A forked worker starts as a copy of this whole kernel, and its garbage collector would walk, and so copy, every object the notebook has made. Freezing them first took the first 32-process run on the test machine from about 4 s to 2.2 s. A script starts small and doesn't need it.

How a worker starts depends on the platform. On Colab (Linux, Python 3.12) it's `fork`: a copy of the running Python, which is quick. Python 3.12 warns that forking a process that runs threads can deadlock (Part 0 hides that warning), and Python 3.14 moved Linux to `forkserver` for that reason. macOS and Windows use `spawn`, a fresh Python each: 32 of those took 9.5 s on the 2-core test machine.

### 2.6 · One process per core, asyncio inside: the scoreboard
*Slide: All 53 Malls, Six Ways.* The production pattern: as many processes as cores, each running the asyncio loop from 2.3 over every Nth mall. Waits overlap in every loop, and every core parses.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from common.malls import SLUGS
from scale.step5_hybrid import shard       # asyncio.run(...) over a list of malls

N = os.cpu_count()                       # one process per core
started = time.perf_counter()
with ProcessPoolExecutor(N) as pool:
    parts = list(pool.map(shard, [SLUGS[k::N] for k in range(N)]))   # every Nth mall
score("A process per core, asyncio inside", started, parts)
print()
scoreboard()

In [ ]:
#@title Optional: the sixth row, 32 threads on Python 3.14t (no GIL). Tick the box, then run. About a minute.
RUN_FREE_THREADED = False  #@param {type:"boolean"}
import subprocess

def uv(*args):
    subprocess.run([sys.executable, "-m", "uv", *args], check=True)

if not RUN_FREE_THREADED:
    print("Skipped. Tick the box and run this cell again for the sixth row.")
else:
    ensure_replay()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    uv("python", "install", "-q", "3.14t")
    uv("venv", "-q", "--allow-existing", "--python", "3.14t", ".venv-ft")
    uv("pip", "install", "-q", "--python", ".venv-ft/bin/python", "requests", "beautifulsoup4")
    run = subprocess.run([".venv-ft/bin/python", "-m", "scale.step2_threads", "--json"],
                         capture_output=True, text=True)
    if run.returncode:
        print(run.stderr[-1500:])
    else:
        row = json.loads(run.stdout.strip().splitlines()[-1])
        print(f"Python {row['python']}, GIL {'on' if row['gil'] else 'off'} during the run")
        record("32 threads, Python 3.14t", row["seconds"], row["rows"])
        print()
        scoreboard()

In [ ]:
#@title Optional: medians of 3 runs each, as on the slides (scale/bench.py, about 2 minutes)
RUN_BENCH = False  #@param {type:"boolean"}
import subprocess

if not RUN_BENCH:
    print("Skipped. Tick the box and run this cell again.")
else:
    ensure_replay()
    started = time.time()
    subprocess.run([sys.executable, "-m", "scale.bench", "--runs", "3"])
    if Path("scale/results.json").exists() and Path("scale/results.json").stat().st_mtime > started:
        print()
        show_results()
    else:
        print("The bench didn't finish: see the error above.")

You can't finish before your slowest page: Opus Mall alone took 1.95 s, so about 2 s is the floor however many workers you add.

## Part 3 · Polite
*Slides: They Set the Bar → What a Site Can See → Your IP Is a Signal*

In [ ]:
#@title Part 3 needs Part 0: check
import os, sys, time, json, warnings
from pathlib import Path
ROOT = next((p for p in (Path("/content/navigating-the-grey"), Path.cwd() / "navigating-the-grey", Path.cwd())
             if (p / "common" / "malls.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run Part 0 (Setup) first.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"   # for !penge
warnings.filterwarnings("ignore", message=r".*fork\(\) may lead to deadlocks")       # see Part 2.5
from notebook_helpers import (HEADERS, ensure_replay, ensure_server, record, score, scoreboard,
                              show_results, timeline)
print("Ready:", ROOT)

### 3.1 · What a limit looks like, on a test server 💻 local only, ~1.5 min
Hold a rate for a few seconds, raise it, and watch what comes back. **This runs against a test server, not a real site:** `limits/ramp.py` starts `replay/server.py --capacity 20`, which we wrote to send at most 20 pages a second and to answer 429 past 25. The plateau you'll see is the one we programmed. The point is the method; never ramp a real site you don't own.

In [ ]:
!{sys.executable} -m limits.ramp

### 3.2 · Your IP is a signal: SM, direct 🌐 1 request
*Slide: Your IP Is a Signal.* The request SM's mall list page makes, with the headers the mall scraper sends, from this machine's own IP. SM refused it from GitHub's cloud servers and from the presenter's home connection. On Colab this machine is a Google server.

In [ ]:
import requests

SM_MALLS = "https://www.smsupermalls.com/list-of-malls"
PARAMS = {"region": "all", "keyword": "", "page": 0}
SCRAPER = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/126.0 Safari/537.36 mallscape-research/1.0",
           "X-Requested-With": "XMLHttpRequest", "Accept": "*/*"}

def malls_in(res):
    """The page's malls and SM's total, or None if this isn't the JSON the scraper expects."""
    try:
        data = res.json()["malls"]
        return data["data"], int(data["counts"])
    except (ValueError, KeyError, TypeError):
        return None

try:
    res = requests.get(SM_MALLS, params=PARAMS, headers=SCRAPER, timeout=30)
    print(f"GET {res.url}   (no proxy: this machine's IP)")
    print(f"  {res.status_code} {res.reason}")
    if res.status_code in (401, 403, 429):
        print("\nSM refused this IP. Same request, same headers: the address is the difference.")
    elif res.ok and malls_in(res):
        print(f"\nSM answered: {malls_in(res)[1]} properties. This IP isn't blocked.")
    elif res.ok:
        print("\nA 200, but not the JSON the scraper expects (a challenge page?). Check every answer, not just the status.")
    else:
        print("\nSM is having trouble right now, which says nothing about your IP. Try later.")
except requests.RequestException as e:
    print(f"No answer from SM ({type(e).__name__}).")

### 3.3 · The same request, through each proxy 🌐 1 request per proxy
A proxy is a middleman: your request leaves from its IP, not yours. In `requests` it's one argument.

**Bring your own.** Webshare's free plan gives 10 datacenter proxies. Download the list as `ip:port:username:password`. In Colab open **🔑 Secrets** in the left sidebar, add a secret named `PROXIES`, paste the list (if it lands on one line, separate the entries with spaces or commas), and turn on notebook access. These cells never print a username, a password or a full address, and they send 3 requests a second in total, however many proxies there are.

In [ ]:
import re

def load_proxies():
    text = os.environ.get("PROXIES", "")
    try:
        from google.colab import userdata
        text = userdata.get("PROXIES") or text
    except Exception:              # not on Colab, no such secret, or notebook access is off
        pass
    proxies = []
    for n, entry in enumerate(re.split(r"[\s,]+", text.strip()), 1):
        parts = entry.split(":")
        if len(parts) != 4:
            if entry:
                print(f"  entry {n}: not ip:port:username:password, skipped")
            continue
        ip, port, user, password = parts
        label = f"#{len(proxies) + 1:02d} {ip.split('.')[0]}.*.*.{ip.split('.')[-1]}"   # never the full address
        url = f"http://{user}:{password}@{ip}:{port}"
        proxies.append((label, {"http": url, "https": url}))
    return proxies

proxies, working = load_proxies(), []
if not proxies:
    print("No PROXIES secret found, or notebook access to it is off: skipping.")
for label, proxy in proxies:
    time.sleep(1 / 3)                          # 3 requests a second across all of them
    try:
        r = requests.get(SM_MALLS, params=PARAMS, headers=SCRAPER, proxies=proxy, timeout=20)
        verdict = f"{r.status_code} {r.reason}"
        if r.ok and malls_in(r):
            working.append((label, proxy))
        elif r.ok:
            verdict += ", but not SM's JSON"
    except requests.RequestException as e:
        verdict = type(e).__name__             # never the message: it can contain the proxy URL
    print(f"  proxy {label:<16} {verdict}")
if proxies:
    print(f"\nSM answered {len(working)} of {len(proxies)} proxies")

### 3.4 · SM's mall list, taking turns across the proxies that got through 🌐 about 11 requests
Every page of the list (12 malls a page), each request from the next working proxy (126 properties on 15 Oct: 11 pages). A request that fails moves on to the next proxy. All of them together still send **3 requests a second**: a proxy changes where you ask from, not how hard you ask.

This is the mall list, which SM's robots.txt allows. The talk's own scraper goes on to the store directories, which robots.txt asks bots to skip, and that's the judgment call on the *Navigating the Grey* slide: not a button in this notebook.

In [ ]:
from itertools import cycle

malls, total, page, failures, next_ok = [], None, 0, 0, 0.0
turns = cycle(working)
while working and failures < 2 * len(working) and page < 20:
    label, proxy = next(turns)
    time.sleep(max(0.0, next_ok - time.monotonic()))
    next_ok = time.monotonic() + 1 / 3         # 3 requests a second, all proxies together
    try:
        r = requests.get(SM_MALLS, params=PARAMS | {"page": page}, headers=SCRAPER, proxies=proxy, timeout=20)
        got = malls_in(r) if r.ok else None
        verdict = f"{r.status_code}"
    except requests.RequestException as e:
        got, verdict = None, type(e).__name__
    if got is None:
        failures += 1
        print(f"  page {page:>2}  via {label:<16} {verdict}: next proxy")
        continue
    batch, total = got
    malls += batch
    print(f"  page {page:>2}  via {label:<16} {verdict}  {len(batch):>2} malls")
    if not batch or len(malls) >= total:
        break
    page += 1

if not working:
    print("No proxy got through in 3.3, so there's nothing to take turns with.")
elif total is None:
    print("\nNo page came back. SM may have changed, or every proxy is blocked now.")
else:
    names = [m["mall_name"].strip() for m in malls if "davao" in m.get("mall_name", "").lower()]
    print(f"\n{len(malls)} of {total} SM properties, {page + 1} pages, {len(working)} proxies, 3 requests a second")
    print("In Davao:", ", ".join(names) or "none in this list")

And a proxy sees every site you visit, so keep it HTTPS, and never send a login through a free one.

## Part 4 · PengePassportPH: the scraping inside the package
*Slides: PengePassportPH → Polite by Construction. A bonus: the talk doesn't run this part on stage.*

PengePassportPH watches DFA passport appointment slots. It looks; it never books. It's a package on PyPI and npm, but the scraping inside it is four requests the booking page itself makes. Here they are by hand, with plain `requests` and BeautifulSoup, one at a time and 3 seconds apart. Then the package, which does the same behind one command.

**Live or offline?** A room of people all asking a government booking site at the same moment is exactly the load the package's limits exist to prevent. So Part 4 starts **offline**: the same code runs against a stand-in on this machine that answers with pages recorded from passport.gov.ph for the package's tests (Antipolo, late Sep to early Oct 2026, so the dates are old). The requests, the token, the empty 200 and the 3-second spacing are real either way. Tick **LIVE** and run Part 4 once, on your own: about 15 requests.

In [ ]:
#@title Part 4 needs Part 0: check
import os, sys, time, json, warnings
from pathlib import Path
ROOT = next((p for p in (Path("/content/navigating-the-grey"), Path.cwd() / "navigating-the-grey", Path.cwd())
             if (p / "common" / "malls.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run Part 0 (Setup) first.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"   # for !penge
warnings.filterwarnings("ignore", message=r".*fork\(\) may lead to deadlocks")       # see Part 2.5
from notebook_helpers import (HEADERS, ensure_replay, ensure_server, record, score, scoreboard,
                              show_results, timeline)
print("Ready:", ROOT)

In [ ]:
#@title Live requests to passport.gov.ph?
LIVE = False  #@param {type:"boolean"}

if LIVE:
    BASE, SITE, SITE_ARG = "https://passport.gov.ph", 19, "davao"     # Davao (SM City Davao)
    os.environ.pop("PENGE_PASSPORT_PH_BASE_URL", None)
    print("LIVE: about 15 requests to passport.gov.ph in this part, 3+ s apart. Run each step once.")
else:
    BASE, SITE, SITE_ARG = "http://127.0.0.1:8802", 486, "486"         # the recordings are Antipolo's
    os.environ["PENGE_PASSPORT_PH_BASE_URL"] = BASE                   # the package reads this in 4.5
    up = ensure_server("passport_standin.py", 8802)
    print("OFFLINE: a stand-in on this machine answers; passport.gov.ph sees nothing." if up
          else "The stand-in didn't start. Run Part 0 again, then this cell.")

### 4.0 · One request at a time, 3 seconds apart
Every request in this part goes through `send()`: it waits its turn, keeps the session's cookie, and stops at a hard cap.

In [ ]:
import random, re
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import requests
from bs4 import BeautifulSoup

GAP = 3.0                    # seconds between requests, like the package
CAP = 30                     # a hard cap for this notebook session
s = requests.Session()       # keeps the cookie between requests
s.headers.update(HEADERS)
next_ok, sent = 0.0, 0

def wait_turn():
    wait = next_ok - time.monotonic()
    if wait > 0:
        print(f"   ⏳ {wait:.1f} s")
        time.sleep(wait)

def send(method, path, **kw):
    """One request at a time, at least GAP seconds apart, plus up to 25% jitter."""
    global next_ok, sent
    if sent >= CAP:
        raise RuntimeError(f"{CAP} requests from this notebook already. That's plenty: stop here.")
    wait_turn()
    sent += 1
    try:
        return s.request(method, BASE + path, timeout=20, **kw)
    finally:
        next_ok = time.monotonic() + GAP * random.uniform(1.0, 1.25)

print(f"{BASE}, site {SITE}: one request at a time, {GAP:g}+ s apart, at most {CAP}")

### 4.1 · The session: a cookie, and a token hidden in the page

In [ ]:
res = send("GET", "/appointment")
page = BeautifulSoup(res.text, "html.parser")
field = page.find("input", {"name": "__RequestVerificationToken"})
if field is None:
    raise RuntimeError("No token on the page: the site has changed.")
token = field["value"]

def script_value(name):                  # a value the page's own JavaScript sets
    m = re.search(name + r"\s*=\s*'([^']*)'", res.text)
    if m is None:
        raise RuntimeError(f"No {name} on the page: the site has changed.")
    return m[1]

today, last_day = script_value("currentDate"), script_value("MAX_DATE")

print(f"GET /appointment   {res.status_code}   {len(res.content) / 1024:.0f} KB of HTML")
print(f"  cookie  {', '.join(c if len(c) < 30 else c[:12] + '…' for c in s.cookies.keys())}")
print(f'  hidden  <input name="__RequestVerificationToken" value="{token[:6]}…">  ({len(token)} characters)')
print(f"  script  currentDate = '{today}'   MAX_DATE = '{last_day}'")

### 4.2 · Ask for the open dates, without the token
The booking page never sends this one. Watch the answer: the site doesn't say no. It says nothing, with a 200.

In [ ]:
DATES = "/appointment/timeslot/available"
AJAX = {"X-Requested-With": "XMLHttpRequest",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"}
form = {"fromDate": today, "toDate": last_day, "siteId": SITE, "requestedSlots": 1}

res = send("POST", DATES, data=form, headers=AJAX)
print(f"POST {DATES}   without the token")
print(f"  {res.status_code} {res.reason}, {len(res.content)} bytes")
if res.status_code == 200 and not res.content:
    print('  ^ a 200 that means "no". Count it as a failure, and back off: 5 s before the next request.')
    next_ok = time.monotonic() + 5

### 4.3 · The same request, with the token
Dates come back as epoch milliseconds at **UTC midnight**. Read them as UTC: in Manila that's 8 AM the same day, but anywhere west of UTC (a consulate in Los Angeles, say) local time lands on the day before.

In [ ]:
res = send("POST", DATES, data=form, headers=AJAX | {"__RequestVerificationToken": token})
days = res.json()                        # one entry per published working day
print(f"POST {DATES}   with the token")
print(f"  {res.status_code} {res.reason}, {len(days)} published days")

def as_date(ms):                         # UTC midnight of the day it means
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).date()

open_days = [as_date(d["AppointmentDate"]) for d in days if d["IsAvailable"]]
if days:
    ms = days[0]["AppointmentDate"]
    print(f"  raw     {json.dumps(days[0], separators=(',', ':'))}")
    print(f"  as UTC  {as_date(ms):%a %d %b %Y}")
    print(f"  read in Manila       {datetime.fromtimestamp(ms / 1000, ZoneInfo('Asia/Manila')):%a %d %b, %H:%M}")
    print(f"  read in Los Angeles  {datetime.fromtimestamp(ms / 1000, ZoneInfo('America/Los_Angeles')):%a %d %b, %H:%M}"
          "   <- a day early")
print(f"\n  open: {len(open_days)} days" + (f", the earliest {open_days[0]:%a %d %b %Y}" if open_days else
      ". Fully booked in the whole window; openings usually appear at 12 noon and 9 PM."))

### 4.4 · The hours of the earliest open day, then stop
An HTML fragment: one radio button per hour, `disabled` when it's booked.

In [ ]:
HOURS = "/appointment/timeslot"
if open_days:
    day = open_days[0].isoformat()
    res = send("POST", HOURS, data={"preferredDate": day, "siteId": SITE, "requiredSlots": 1},
               headers=AJAX | {"__RequestVerificationToken": token})
    print(f"POST {HOURS}   {day}")
    print(f"  {res.status_code} {res.reason}")
    for label in BeautifulSoup(res.text, "html.parser").select("label"):
        radio, hours = label.find("input"), label.find(string=re.compile(r"\d+:\d+\s*-\s*\d+:\d+"))
        status = label.select_one(".col-xs-5")
        if radio and hours and status:
            print(f"  {hours.strip()}  {'open  ' if not radio.has_attr('disabled') else 'booked'}  "
                  f"{' '.join(status.get_text().split())}")
else:
    print("No open day to ask about.")
print("\nWe stop here. Picking an hour and pressing NEXT holds it for 30 minutes, and that step is")
print("behind reCAPTCHA. The package never takes it, and never keeps the slot ids (each radio's value).")

(Offline, the hours recorded for 5 Oct stand in for every date.)

### 4.5 · The same four requests, in one command
`penge check --site davao --times` (offline: `--site 486`, Antipolo). The package adds what the cells above only sketch: a limiter you can't switch off (3+ s apart with jitter, an hourly budget, back-off, a circuit breaker), session refresh, and checks on every answer.

In [ ]:
wait_turn()                              # 3 s after our own last request
!penge check --site {SITE_ARG} --times        # LIVE: davao. Offline: 486, Antipolo

### 4.6 · Two processes, one budget
Two `penge check` processes start at the same moment. Every process on this machine shares one locked state file, so together they still never go faster than one request every 3 seconds.

In [ ]:
import subprocess, threading, urllib.parse
from penge_passport_ph.rate_limit import default_state_dir

since = time.time() * 1000
lock = threading.Lock()

def run(label, site):
    cmd = ["penge", "check", "--site", site]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=os.environ | {"PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        if line.strip():
            with lock:
                print(f"[{label}] {line.rstrip()}")
    proc.wait()

pair = (("A", "19"), ("B", "486")) if LIVE else (("A", "486"), ("B", "486"))   # Davao, Antipolo
for label, site in pair:
    print(f"[{label}] $ penge check --site {site}")
threads = [threading.Thread(target=run, args=a) for a in pair]
for th in threads:
    th.start()
for th in threads:
    th.join()

host = urllib.parse.urlsplit(BASE).netloc
state = default_state_dir() / ("".join(c if c.isalnum() or c in ".-" else "_" for c in host) + ".json")
when = sorted(t for t in json.loads(state.read_text())["dispatched"] if t >= since) if state.exists() else []
print(f"\nWhat {state.name} recorded: when each request left, from both processes")
gaps = [(b - a) / 1000 for a, b in zip(when, when[1:])]
for i, ms in enumerate(when):
    print(f"  request {i + 1}   +{(ms - when[0]) / 1000:5.1f} s" + (f"   {gaps[i - 1]:.1f} s after the one before" if i else ""))
if gaps:
    print(f"\n2 processes, {len(when)} requests, never closer than {min(gaps):.1f} s apart.")

## Part 5 · Run it anywhere
Colab can't run Docker, but this is all a scraper with a browser needs to run the same on any cloud. Playwright's official image already has Chromium and Xvfb inside:

```dockerfile
# Dockerfile: Chromium and Xvfb come built in
FROM mcr.microsoft.com/playwright/python:v1.63.0
RUN pip install playwright==1.63.0
COPY . /app
CMD ["xvfb-run", "python", "/app/run.py"]
```

Run it with `--ipc=host` (or Chromium can run out of shared memory) and `--init`. A new cloud gives you new IP ranges, not a new rate limit.

## Where to go next
- **Philippine Mall Explorer**: 41,814 stores across 10 mall chains, rebuilt monthly. [github.com/alpharomercoma/philippine-mall-explorer](https://github.com/alpharomercoma/philippine-mall-explorer)
- **PengePassportPH**: [github.com/alpharomercoma/penge-passport-ph](https://github.com/alpharomercoma/penge-passport-ph)

Salamat, Davao!